<a href="https://colab.research.google.com/github/jiraroj-wir/MUIC-ICCS261-Principles-of-Data-Science/blob/main/Copy_of_loading_legacy_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading Legacy Data

*K. Bunchongchit<br>
Last updated on September 24, 2026*

Data in this task contains the daily weather data from the Global Historical Climatology Network for one weather station (MX17004) in Mexico from **1955 to 2011**. The data set has a column for each possible day in the month, and the "element" column indicates if that row contains the minimum or maximum temperature or the amount of percipitation.

Let's load the data file.

***Warning***: Code in this part will be messy as it is showing the actual process of figuring out a solution. Most of the time we show only the "success". So it is not usual to see how chaotic it could be behind the scence. Realizing this will make you more aware when asking the data engineer team to do something because a regular request may not be that simple.

If you are using a desktop IDE, try
<pre>df_wt = pd.read_csv('data/weather.txt', header=0, sep='\t', engine='python')</pre>with the adjusted folder path.

In [ ]:
import pandas as pd

weather_URL = 'https://drive.google.com/uc?export=download&id=' + '1d9b_at4SJMlgtzQ04isPjUVzGEm7hYsl'
df_wt = pd.read_csv(weather_URL)
df_wt

There are many problems with this dataset.
- The first data row is turned to be the column labels.
- There is only one column of data.
- Column separators are inconsistent throughout the file, somes are "I", some are "S", and some are just blanks.


So let's check the raw text file to confirm these issues. We will examine some lines at the beginning and the end of this data file.

In [ ]:
# https://stackoverflow.com/questions/1393324/given-a-url-to-a-text-file-what-is-the-simplest-way-to-read-the-contents-of-the
import urllib3  # the lib that handles the url stuff

http = urllib3.PoolManager()
response = http.request('GET', weather_URL)
data = response.data.decode('utf-8')

lines = data.split('\n')
print(f'no of lines = {len(lines)}')
print('The first three rows')
print('\n'.join(lines[:3]))
print('The last three rows')
print('\n'.join(lines[-4:]))  # The actual last line is empty

So our observation is correct; the column separators are inconsistent. As the raw file does not contain "\t", our data is not tab-limited either. So we should try the "I" as a separator next. Even though the separators are inconsistent, trying a simple thing normally gives clues what to do next.

However, before we go on and trying to detangle this mess. Let's get a good understand the data in the file.

In [ ]:
df_wt = pd.read_csv(weather_URL, header=0, sep=' I ')
df_wt

Nope, it fails earlier than expected since Line 5.


What else can I try? As this is a legacy data file, could it be that the typical delimeter convention did not apply? So let's use a string as a delimiter. Of course, this cannot be the only delimiter, but as stated before, let's try simple things first.

In [ ]:
df_wt = pd.read_fwf(weather_URL, sep=' I ')
df_wt

Surprisingly, everything is fit to columns even with line with no "I" as the delimiter. After looking at the raw text again, it could be a fixed-width text file.


Let's try this with a few columns first.

In [ ]:
# Warning: Experimental code in progress!
# https://towardsdatascience.com/parsing-fixed-width-text-files-with-pandas-f1db8f737276

colspacing = [(0, 17), (17, 21), (21, 29), (29, None)]
df_temp = pd.read_fwf(weather_URL, colspecs=colspacing, header=None,
            names=['station', 'measurement', 'd1', 'rest'])
df_temp

This is even worse. Now we have the letter 'I' mixed into the values, and the first row of data turn into column labels. OK, let's go for another try without column specifications. Keep fingers crossed while running this!

In [ ]:
df_wt = pd.read_fwf(weather_URL, names=['Description']+['c'+ str(i) for i in range(1,64)])
df_wt

Whew... The dataframe now seems to have columns nicely separated. But there are still many junk columns. I manage to find another use of ```pandas.fwf``` from https://sparkbyexamples.com/pandas/pandas-read-text-into-dataframe/. With some more calculations, we generate the column headers and then the array to speicify width of each column.

In [ ]:
# https://stackoverflow.com/questions/952914/how-do-i-make-a-flat-list-out-of-a-list-of-lists
import functools
import itertools
import operator

cols = ['station', 'month', 'type'] + functools.reduce(operator.iconcat, [['d'+str(i), 'u' + str(i)] for i in range(1, 32)], [])
print(cols)

In [ ]:
# https://sparkbyexamples.com/pandas/pandas-read-text-into-dataframe/
df_wt = pd.read_fwf(weather_URL,  # the URL from which the data file is being read.
                    header=None,  # This tells pandas that the file does not have a header row
                    widths=[11,6,4] + [7, 1]*31,  # See below
                    names=cols)  # custom column names generated in the previous code cell
df_wt

# This is a list that defines the width of each column in the fixed-width file.
# It specifies the width for the first three columns as 11, 6, and 4 characters
# respectively, and then repeats a pattern of 7 and 1 characters for the next
# 31 pairs of columns. This is crucial for correctly parsing the data into separate columns.

Now it is a successful attmept even though not perfect. Data are in their own columns. There are still two issues to deal with though.
1. There is some mess in the delimeter columns but we are going to discard that anyway.
2. Now the sentinel value from missing data has changed from -9999 to 999. We can change it to NaN. The reason for selecting this value will be discussed on the next notebook.

We will drop the delimeter columns first.

In [ ]:
df_wt.drop(['u'+str(i) for i in range(1, 32)], axis=1, inplace=True)
df_wt

In [ ]:
df_wt.info()

In [ ]:
for col_name in df_wt.columns[3:]:
    # Convert to numeric first, coercing any non-numeric values to NaN
    df_wt[col_name] = pd.to_numeric(df_wt[col_name], errors='coerce')
    # Then convert to nullable integer type to handle NaN values
    df_wt[col_name] = df_wt[col_name].astype('Int64')

df_wt

In [ ]:
df_wt.info()

Lastly, we fix the sentinel value to be the popular choice for DataFrame.

In [ ]:
import numpy as np

df_wt.replace(-9999, np.nan, inplace=True)
df_wt